# Kapitel 3 - Koduppgift 14

In [2]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


# läs in data
diabetes = load_diabetes(as_frame=True)
df = diabetes.frame

# visa de första 5 raderna för att få en överblick
print("Första 5 raderna i datasetet:")
print(df.head())
print()

Första 5 raderna i datasetet:
        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  target  
0 -0.002592  0.019907 -0.017646   151.0  
1 -0.039493 -0.068332 -0.092204    75.0  
2 -0.002592  0.002861 -0.025930   141.0  
3  0.034309  0.022688 -0.009362   206.0  
4 -0.002592 -0.031988 -0.046641   135.0  



In [3]:
# dela upp i features (X) och target (y)
# x oberoende variabler
# y beroende variabel (det vi vill prediktera)
X = df.drop(columns='target')   
y = df['target'] 

In [4]:
# dela av 20% som testdata (används inte förrän på slutet)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42 )

# dela resterande 80% i träningsdata (60%) och valideringsdata (20%) , 25% av 80% = 20% av totalen
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42)


print(f"Träningsdata: {X_train.shape[0]} observationer")
print(f"Valideringsdata: {X_val.shape[0]} observationer")
print(f"Testdata: {X_test.shape[0]} observationer")
print()

Träningsdata: 264 observationer
Valideringsdata: 89 observationer
Testdata: 89 observationer



In [5]:
# 2 olika modeller 
linreg = LinearRegression()
tree = DecisionTreeRegressor(random_state=42) 

print("Tränar modeller")
linreg.fit(X_train, y_train)
tree.fit(X_train, y_train)
print("Klar!")
print()

Tränar modeller
Klar!



In [6]:
# utvärdera modellerna på valideringsdatan
rmse_linreg = root_mean_squared_error(y_val, linreg.predict(X_val))
rmse_tree = root_mean_squared_error(y_val, tree.predict(X_val))

print("Utvärdering på valideringsdata:")
print(f"Linjär regression - RMSE: {rmse_linreg:.2f}")
print(f"Beslutsträd      - RMSE: {rmse_tree:.2f}")
print()

Utvärdering på valideringsdata:
Linjär regression - RMSE: 51.19
Beslutsträd      - RMSE: 65.81



In [7]:
# välj den modell som har lägst RMSE
if rmse_linreg < rmse_tree:
    best_model = linreg
    best_model_name = "Linjär Regression"
else:
    best_model = tree
    best_model_name = "Beslutsträd"

print(f"Bästa modell: {best_model_name}")
print()

Bästa modell: Linjär Regression



In [8]:
# träna om den valda modellen
print(f"Tränar om {best_model_name}")
best_model.fit(X_train_full, y_train_full)
print("Klar!")
print()

Tränar om Linjär Regression
Klar!



In [9]:
# test på osedd data
y_test_pred = best_model.predict(X_test)
rmse_test = root_mean_squared_error(y_test, y_test_pred)

print("Utverdäring på testdata:")
print(f"Modell: {best_model_name}")
print(f"RMSE: {rmse_test:.2f}")
print()

Utverdäring på testdata:
Modell: Linjär Regression
RMSE: 53.85



#### Förbättreing med GridGridSearch

In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV


# definiera vilka hyperparametrar som ska testas
# n_estimators = antal träd i skogen
# max_depth = maximalt djup på träden 

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10, None]
}


In [11]:
# skapa grid search med 5-delad korsvalidering

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

In [12]:
# kör grid search på tränings + valideringsdata
grid_search.fit(X_train_full, y_train_full)

# hämta den bästa modellen från grid search
best_rf = grid_search.best_estimator_

# utvärdera den optimerade modellen på testdata
rmse_rf_test = root_mean_squared_error(y_test, best_rf.predict(X_test))

print(f"RMSE på testdata med Random Forest: {rmse_rf_test:.2f}")

RMSE på testdata med Random Forest: 53.48


##### Man kan ändra hyperparametrar på param_grid för att få ett bättre värde, men testningen tar för lång tid. 